# Hospital Readmission Prediction Analysis

This notebook performs:
1. Hyperparameter tuning with out-of-time validation
2. Cross-validation with optimized parameters
3. SHAP analysis for model interpretability
4. Partial dependence plots for top features

In [ ]:
# Cell 1: Setup
import os

OUTPUT_DIR = './shap_results'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Cell 2: Configuration Constants
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import FunctionTransformer, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import cross_validate
from sklearn.metrics import roc_auc_score, make_scorer, classification_report, confusion_matrix
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import shap

# Import shared utilities from model_utils
from model_utils import (
    DateBasedTimeSeriesSplitter,
    create_preprocessing_pipeline,
    calculate_tuning_cutoff,
    generate_classification_report,
    generate_confusion_matrix,
    generate_lift_table,
    generate_subgroup_auc_report,
    tune_hyperparameters,
)

# Data configuration
DATE_COLUMN = 'date'
DATA_FILE = 'diabetic_data_with_dates.csv'
TARGET_COLUMN = 'readmitted'
TARGET_MAPPING_LOGIC = lambda x: 0 if x == 'NO' else 1
CATEGORICAL_MISSING_VALUE = -1
NUMERICAL_MISSING_VALUE = -9999

# Cross-validation configuration
CV_CONFIG = {
    'window': 530,        # 18 months training window
    'fh': 30,             # 30 days forecast horizon
    'test_window': 60,    # 60 days test window
    'step': 90,           # 90 days step length
    'n_splits': 5         # 5 CV folds
}

# Hyperparameter tuning constants
TUNING_N_SPLITS = 3      # Fewer folds for tuning (speed)
TUNING_N_TRIALS = 30     # Number of FLAML trials
TUNING_TIMEOUT = 1800    # 30 minute timeout
EARLY_STOPPING_ROUNDS = 50
MAX_N_ESTIMATORS = 1000  # High value, early stopping finds optimal

# Class names
NEGATIVE_CLASS_NAME = 'No Readmission'
POSITIVE_CLASS_NAME = 'Readmission'
SUBGROUP_ANALYSIS_FEATURE = 'race'  # Feature for AUC subgroup analysis

# Feature definitions
CATEGORICAL_FEATURES = [
    'A1Cresult', 'acarbose', 'acetohexamide', 'age', 'change', 'chlorpropamide',
    'citoglipton', 'diabetesMed', 'diag_1', 'diag_2', 'diag_3', 'examide',
    'gender', 'glimepiride', 'glimepiride-pioglitazone', 'glipizide',
    'glipizide-metformin', 'glyburide', 'glyburide-metformin', 'insulin',
    'max_glu_serum', 'medical_specialty', 'metformin', 'metformin-pioglitazone',
    'metformin-rosiglitazone', 'miglitol', 'nateglinide', 'payer_code',
    'pioglitazone', 'race', 'repaglinide', 'rosiglitazone', 'tolazamide',
    'tolbutamide', 'troglitazone', 'weight',
]

NUMERICAL_FEATURES = [
    'admission_source_id', 'admission_type_id', 'discharge_disposition_id',
    'num_lab_procedures', 'num_medications', 'num_procedures', 'number_diagnoses',
    'number_emergency', 'number_inpatient', 'number_outpatient', 'time_in_hospital',
]

FEATURE_NAMES = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

print(f"Total features: {len(FEATURE_NAMES)} ({len(CATEGORICAL_FEATURES)} categorical, {len(NUMERICAL_FEATURES)} numerical)")

In [ ]:
# Cell 3: Data Loading

def load_data(file_path, date_column, target_column):
    """Load and preprocess the dataset."""
    print("Loading data...")
    df = pd.read_csv(file_path)
    df[date_column] = pd.to_datetime(df[date_column])
    df['target'] = df[target_column].apply(TARGET_MAPPING_LOGIC)
    df = df.sort_values(date_column).reset_index(drop=True)

    print(f"Dataset shape: {df.shape}")
    print(f"Date range: {df[date_column].min()} to {df[date_column].max()}")
    print(f"Target distribution: {df['target'].value_counts().to_dict()}")
    return df


# Load data
df = load_data(DATA_FILE, DATE_COLUMN, TARGET_COLUMN)
X = df[FEATURE_NAMES].copy()
y = df['target'].copy()


In [ ]:
# Cell 4: Preprocessing Pipeline Setup
# Functions imported from model_utils: create_preprocessing_pipeline, generate_lift_table, generate_subgroup_auc_report

print("Preprocessing functions loaded from model_utils.")


In [ ]:
# Cell 5: Hyperparameter Tuning with FLAML (Out-of-Time Validation)
# Functions imported from model_utils: calculate_tuning_cutoff, tune_hyperparameters

# Calculate tuning cutoff
cv_config = {
    'window': CV_CONFIG['window'],
    'fh': CV_CONFIG['fh'],
    'test_window': CV_CONFIG['test_window'],
    'step': CV_CONFIG['step'],
    'n_splits': CV_CONFIG['n_splits']
}

tuning_cutoff = calculate_tuning_cutoff(df, DATE_COLUMN, cv_config)
print(f"Main CV starts training from: {tuning_cutoff + pd.Timedelta(days=30)}")
print(f"Tuning data ends at: {tuning_cutoff}")

# Filter data for tuning
df_tuning = df[df[DATE_COLUMN] <= tuning_cutoff].copy()
X_tuning = df_tuning[FEATURE_NAMES].copy()
y_tuning = df_tuning['target'].values

print(f"Tuning dataset shape: {df_tuning.shape}")
print(f"Tuning date range: {df_tuning[DATE_COLUMN].min()} to {df_tuning[DATE_COLUMN].max()}")

# Pre-transform data ONCE for tuning efficiency
print("\nPre-transforming data for tuning...")
preprocessing_for_tuning = create_preprocessing_pipeline(
    CATEGORICAL_FEATURES,
    NUMERICAL_FEATURES,
    CATEGORICAL_MISSING_VALUE,
    NUMERICAL_MISSING_VALUE
)
X_tuning_transformed = preprocessing_for_tuning.fit_transform(X_tuning)
print(f"Transformed shape: {X_tuning_transformed.shape}")

# Run hyperparameter tuning
best_params, best_tuning_score, study = tune_hyperparameters(
    X_tuning_transformed, 
    y_tuning,
    n_splits=TUNING_N_SPLITS,
    n_trials=TUNING_N_TRIALS,
    timeout=TUNING_TIMEOUT
)


In [ ]:
# Cell 6: Cross-Validation with Tuned Parameters

print("Setting up main cross-validation with tuned parameters...")

# Create splitter for main CV
main_splitter = DateBasedTimeSeriesSplitter(
    window_length=CV_CONFIG['window'],
    fh=CV_CONFIG['fh'],
    test_window_length=CV_CONFIG['test_window'],
    step_length=CV_CONFIG['step'],
    n_splits=CV_CONFIG['n_splits']
)

splits = list(main_splitter.split(df, date_column=DATE_COLUMN))
print(f"Generated {len(splits)} main CV splits")

# Display split date ranges
print("\nMain CV Split Date Ranges:")
for fold_num, (train_idx, test_idx) in enumerate(splits, 1):
    train_dates = df[DATE_COLUMN].iloc[train_idx]
    test_dates = df[DATE_COLUMN].iloc[test_idx]
    print(f"Fold {fold_num}: Train {train_dates.min().date()} to {train_dates.max().date()} | "
          f"Test {test_dates.min().date()} to {test_dates.max().date()} | "
          f"Train: {len(train_idx):,}, Test: {len(test_idx):,}")

# Create tuned model with best parameters
tuned_model = XGBClassifier(
    tree_method='hist',
    n_estimators=best_params['n_estimators'],
    learning_rate=best_params['learning_rate'],
    max_depth=best_params['max_depth'],
    min_child_weight=best_params['min_child_weight'],
    subsample=best_params['subsample'],
    colsample_bytree=best_params['colsample_bytree'],
    reg_lambda=best_params['reg_lambda'],
)

# Create pipeline with tuned model
optimized_pipeline = Pipeline([
    ('preprocessing', create_preprocessing_pipeline(CATEGORICAL_FEATURES, NUMERICAL_FEATURES, CATEGORICAL_MISSING_VALUE, NUMERICAL_MISSING_VALUE)),
    ('model', tuned_model)
])

# Run cross-validation with train scores
print("\nRunning cross-validation with tuned parameters...")
cv_results = cross_validate(
    estimator=optimized_pipeline,
    X=X,
    y=y,
    cv=splits,
    scoring={'roc_auc': make_scorer(roc_auc_score, response_method='predict_proba')},
    return_estimator=True,
    return_train_score=True,
    verbose=1
)

# Extract AUC scores
auc_scores = cv_results['test_roc_auc']
train_auc_scores = cv_results['train_roc_auc']

# Display Train vs Test AUC
print("\n" + "="*60)
print("CROSS-VALIDATION RESULTS")
print("="*60)
print(f"\n{'Fold':<6} {'Train AUC':<12} {'Test AUC':<12}")
print("-" * 30)
for i, (train_auc, test_auc) in enumerate(zip(train_auc_scores, auc_scores), 1):
    print(f"{i:<6} {train_auc:<12.4f} {test_auc:<12.4f}")
print("-" * 30)
print(f"{'Avg':<6} {np.mean(train_auc_scores):<12.4f} {np.mean(auc_scores):<12.4f}")
print(f"{'Std':<6} {np.std(train_auc_scores):<12.4f} {np.std(auc_scores):<12.4f}")

# Collect predictions and generate lift tables by fold
print("\n" + "="*80)
print("LIFT ANALYSIS BY FOLD")
print("="*80)

all_y_true = []
all_y_pred = []
all_y_proba = []
all_subgroup_values = []

for fold_num, ((train_idx, test_idx), estimator) in enumerate(zip(splits, cv_results['estimator']), 1):
    X_test_fold = X.iloc[test_idx]
    y_test_fold = y.iloc[test_idx]
    y_pred_fold = estimator.predict(X_test_fold)
    y_proba_fold = estimator.predict_proba(X_test_fold)[:, 1]
    
    # Generate lift table for this fold
    generate_lift_table(y_test_fold.values, y_proba_fold, fold_num=fold_num, positive_class_name=POSITIVE_CLASS_NAME)
    
    all_y_true.extend(y_test_fold)
    all_y_pred.extend(y_pred_fold)
    all_y_proba.extend(y_proba_fold)
    all_subgroup_values.extend(df.iloc[test_idx][SUBGROUP_ANALYSIS_FEATURE].values)

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_y_proba = np.array(all_y_proba)
all_subgroup_values = np.array(all_subgroup_values)

print(f"\n{'='*80}")
print(f"Total predictions collected across all folds: {len(all_y_pred)}")
print(f"{'='*80}")

print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(all_y_true, all_y_pred, 
                           target_names=[NEGATIVE_CLASS_NAME, POSITIVE_CLASS_NAME]))

print("\n" + "="*60)
print("CONFUSION MATRIX")
print("="*60)
cm = confusion_matrix(all_y_true, all_y_pred)
print(f"\n                    Predicted {NEGATIVE_CLASS_NAME:<15} Predicted {POSITIVE_CLASS_NAME}")
print(f"Actual {NEGATIVE_CLASS_NAME:<13} {cm[0, 0]:8d}        {cm[0, 1]:8d}")
print(f"Actual {POSITIVE_CLASS_NAME:<13} {cm[1, 0]:8d}        {cm[1, 1]:8d}")

# Subgroup AUC analysis
generate_subgroup_auc_report(all_y_true, all_y_proba, all_subgroup_values, subgroup_name=SUBGROUP_ANALYSIS_FEATURE)


In [ ]:
# Cell 7: SHAP Analysis (Final Fold)

print("Performing SHAP analysis on final fold...")

# Get final fold model and data
final_estimator = cv_results['estimator'][-1]
final_test_idx = splits[-1][1]
X_test_final = X.iloc[final_test_idx]
y_test_final = y.iloc[final_test_idx]

print(f"Final fold test set size: {len(X_test_final):,} samples")
print(f"Final fold date range: {df.iloc[final_test_idx][DATE_COLUMN].min().date()} to {df.iloc[final_test_idx][DATE_COLUMN].max().date()}")

# Transform features through preprocessing pipeline
X_transformed = final_estimator.named_steps['preprocessing'].transform(X_test_final)

# Create explainer and compute SHAP values
print("Computing SHAP values...")
explainer = shap.TreeExplainer(final_estimator.named_steps['model'])
shap_values = explainer(X_transformed)

# Assign feature names to the SHAP Explanation object for proper labeling in plots
shap_values.feature_names = FEATURE_NAMES

# Store SHAP data for later use
shap_data = {
    'shap_values': shap_values.values,
    'base_value': shap_values.base_values,
    'feature_values': X_transformed,
    'feature_names': FEATURE_NAMES,
    'y_true': y_test_final.values,
    'y_pred_proba': final_estimator.predict_proba(X_test_final)[:, 1]
}

print(f"SHAP values shape: {shap_values.values.shape}")

# Calculate feature importance from SHAP
shap_importance = np.abs(shap_values.values).mean(axis=0)
shap_importance_df = pd.DataFrame({
    'Feature': FEATURE_NAMES,
    'SHAP_Importance': shap_importance
}).sort_values('SHAP_Importance', ascending=False)

print("\nTop 10 Features by SHAP Importance:")
print(shap_importance_df.head(10).to_string(index=False))

In [ ]:
# Cell 8: SHAP Visualizations

# Get top 10 features for visualization
top_10_features = shap_importance_df.head(10)['Feature'].tolist()
top_10_indices = [FEATURE_NAMES.index(f) for f in top_10_features]

# SHAP Summary Plot (Beeswarm)
print("Generating SHAP Summary Plot...")
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(
    shap_values.values, 
    X_transformed, 
    feature_names=FEATURE_NAMES,
    max_display=15,
    show=False
)
plt.title('SHAP Feature Importance (Final Fold)', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_summary.png', dpi=150, bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/shap_summary.png")
plt.show()

# SHAP Bar Plot
print("\nGenerating SHAP Bar Plot...")
fig, ax = plt.subplots(figsize=(10, 8))
shap.plots.bar(shap_values, max_display=15, show=False)
plt.title('Mean |SHAP Value| by Feature', fontsize=14)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/shap_bar.png', dpi=150, bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/shap_bar.png")
plt.show()

# SHAP Dependence Plots for Top 5 Features
print("\nGenerating SHAP Dependence Plots for Top 5 Features...")
for i, (feat_name, feat_idx) in enumerate(zip(top_10_features[:5], top_10_indices[:5])):
    fig, ax = plt.subplots(figsize=(8, 6))
    shap.dependence_plot(
        feat_idx,
        shap_values.values,
        X_transformed,
        feature_names=FEATURE_NAMES,
        show=False,
        ax=ax
    )
    plt.title(f'SHAP Dependence: {feat_name}', fontsize=12)
    plt.tight_layout()
    
    filename = f'shap_dependence_{i+1}_{feat_name.replace("-", "_")}.png'
    plt.savefig(f'{OUTPUT_DIR}/{filename}', dpi=150, bbox_inches='tight')
    print(f"Saved to {OUTPUT_DIR}/{filename}")
    plt.show()

print("SHAP visualizations complete!")

In [ ]:
# Cell 9: Partial Dependence Plots & Save Results

from sklearn.inspection import PartialDependenceDisplay
import pickle
import json as json_module

# Partial Dependence Plots for Top 10 Features
print("Generating Partial Dependence Plots for Top 10 Features...")

# Create 2x5 grid for top 10 features
fig, axes = plt.subplots(2, 5, figsize=(20, 8))

# PDP needs to work on the XGBoost model directly with transformed data
PartialDependenceDisplay.from_estimator(
    final_estimator.named_steps['model'],
    X_transformed,
    features=top_10_indices,
    feature_names=FEATURE_NAMES,
    ax=axes.ravel(),
    kind='average',
    n_cols=5
)

plt.suptitle('Partial Dependence Plots - Top 10 Features', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/partial_dependence.png', dpi=150, bbox_inches='tight')
print(f"Saved to {OUTPUT_DIR}/partial_dependence.png")
plt.show()

# Save Results
print("\n" + "="*60)
print("SAVING RESULTS")
print("="*60)

# Prepare metadata
metadata = {
    'fold': 5,
    'n_samples': len(y_test_final),
    'date_range': [
        str(df.iloc[final_test_idx][DATE_COLUMN].min().date()),
        str(df.iloc[final_test_idx][DATE_COLUMN].max().date())
    ],
    'best_params': {k: (int(v) if isinstance(v, (np.integer, int)) else float(v) if isinstance(v, (np.floating, float)) else v) 
                   for k, v in best_params.items()},
    'auc_scores': [float(s) for s in auc_scores],
    'mean_auc': float(np.mean(auc_scores)),
    'std_auc': float(np.std(auc_scores)),
    'top_10_features_shap': top_10_features
}

# Save SHAP data as pickle
with open(f'{OUTPUT_DIR}/shap_data.pkl', 'wb') as f:
    pickle.dump(shap_data, f)

# Save metadata as JSON
with open(f'{OUTPUT_DIR}/metadata.json', 'w') as f:
    json_module.dump(metadata, f, indent=2)

print(f"Results saved to: {OUTPUT_DIR}")

print("\nMetadata:")
print(json_module.dumps(metadata, indent=2))

print("\n" + "="*60)
print("ANALYSIS COMPLETE")
print("="*60)